# JobSpy manual check

Interactive tool for debugging why `POST /scrape/jobspy` (backed by `app/sources/jobspy_source.py`) returns few/no results, independent of the FastAPI app or Docker — talks to `jobspy`/the job sites directly.

**Run this with the project's `venv` as the kernel** (Kernel → Change Kernel → the `careerops`/`venv` interpreter at `../venv`), since `jobspy`, `pandas`, and the app's own modules are installed there, not globally.

## Already found & fixed (2026-09-19)

`jobspy.scrape_jobs()` defaults `country_indeed="usa"`. This app is India-only
(`data/constraints.yaml`'s `allowed_locations`: Remote/Bangalore/Pune/Hyderabad), and that
one setting is shared by **both** the Indeed and Glassdoor scrapers internally
(`jobspy.model.Country.from_string(country_indeed)`) — so every India-location search was
silently querying Indeed's *USA* catalog and returning nothing. Fixed in
`app/sources/jobspy_source.py` (`DEFAULT_COUNTRY = "india"`, applied via `kwargs.setdefault(...)`
so a caller can still override it). Verified below: Indeed goes from 0 rows → real results.

## Still flaky/blocked as of this check — not a code bug, site-side anti-scraping

- **Glassdoor**: `findPopularLocationAjax.htm` (its own location-autocomplete lookup) returns
  HTTP 400 regardless of country — looks like it's blocking this network/IP, not a bad location
  string.
- **Naukri**: `406 recaptcha required` — needs a real browser/captcha solve, not fixable by
  changing request params.
- **ZipRecruiter**: `403 forbidden` — same class of block.
- **Google**: 0 rows, no error surfaced — inconclusive from here; try different phrasing/timing.

None of these raise inside `app/sources/jobspy_source.py:fetch_jobs()` — jobspy logs the error
and that site just contributes 0 rows to the merged DataFrame, which is why `/scrape/jobspy`
returns a clean `{"found": 0}` instead of a 500. Re-run the cells below any time to see current
status — blocking behavior can change run to run (rate limits, IP reputation, etc.).

In [ ]:
import sys
from pathlib import Path

# So `import app...` resolves the same as it does when uvicorn runs from the repo root.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import logging
logging.basicConfig(level=logging.INFO)

import pandas as pd
pd.set_option("display.max_colwidth", 80)

from jobspy import scrape_jobs

SEARCH_TERM = "backend engineer"
LOCATION = "Bangalore"
print("ready")

## 1. Reproduce the original bug: Indeed with the library's default country ("usa")

In [ ]:
df_usa = scrape_jobs(
    site_name=["indeed"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=5,
    country_indeed="usa",  # jobspy's own default — this is the bug
)
print("rows (country=usa):", len(df_usa) if df_usa is not None else None)

## 2. Same query with the fixed default ("india") — should return real rows

In [ ]:
df_india = scrape_jobs(
    site_name=["indeed"],
    search_term=SEARCH_TERM,
    location=LOCATION,
    results_wanted=5,
    country_indeed="india",
)
print("rows (country=india):", len(df_india) if df_india is not None else None)
if df_india is not None and not df_india.empty:
    display(df_india[["title", "company", "site", "location", "job_url"]])

## 3. Through the app's own wrapper (`app/sources/jobspy_source.fetch_jobs`)

This is exactly what `POST /scrape/jobspy` calls — confirms the fixed default is actually wired
up, and shows the normalized (DB-ready) shape instead of jobspy's raw DataFrame columns.

In [ ]:
from app.sources.jobspy_source import fetch_jobs

jobs = fetch_jobs(SEARCH_TERM, location=LOCATION, sites=["indeed"], results_wanted=5)
print("normalized jobs:", len(jobs))
pd.DataFrame(jobs)[["source", "company", "title", "location", "url"]] if jobs else "no rows"

## 4. Sweep every allowed site individually

One site per call (not a combined multi-site call) so one blocked site's error doesn't obscure
another's result count. Re-run this cell any time to get current live status per site.

In [ ]:
from app.sources.jobspy_source import ALLOWED_SITES

results = {}
for site in ALLOWED_SITES:
    try:
        df = scrape_jobs(
            site_name=[site],
            search_term=SEARCH_TERM,
            location=LOCATION,
            results_wanted=5,
            country_indeed="india",
        )
        results[site] = len(df) if df is not None else 0
    except Exception as exc:
        results[site] = f"raised: {exc!r}"

for site, outcome in results.items():
    print(f"{site:14s} -> {outcome}")

## 5. Google — try with an explicit `google_search_term`

jobspy's Google scraper reads a Google-for-Jobs-style natural-language query separately from
`search_term`; an explicit one sometimes surfaces results a bare keyword search doesn't.

In [ ]:
df_google = scrape_jobs(
    site_name=["google"],
    search_term=SEARCH_TERM,
    google_search_term=f"{SEARCH_TERM} jobs near {LOCATION} since yesterday",
    location=LOCATION,
    results_wanted=5,
    country_indeed="india",
)
print("rows:", len(df_google) if df_google is not None else None)
if df_google is not None and not df_google.empty:
    display(df_google[["title", "company", "site", "location"]])

## 6. Full app-level call, exactly as `POST /scrape/jobspy` makes it

Sanity check against every allowed site at once, the way the real route calls it.

In [ ]:
jobs_all = fetch_jobs(SEARCH_TERM, location=LOCATION, results_wanted=20)
print("total normalized jobs across all sites:", len(jobs_all))
pd.DataFrame(jobs_all)[["source", "company", "title", "location"]] if jobs_all else "no rows"